In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle #this is important so that we can reuse this file later

In [2]:
#Load Dataset
data=pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
# first 3 features rownum,customerid,surname aren't that important

#Preprocess Data
## Drop Irrelevant columns, axis =1 means column
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)

In [4]:
# data

# Geography and Gender are categorical employee and we can apply encoding

label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])
# data (we will see that the Gender will be 0 or 1 depending on Male or Female)

In [5]:
## One Hot encoode geography column

from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo=OneHotEncoder()
geo_encoder=onehot_encoder_geo.fit_transform(data[['Geography']])
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [6]:
onehot_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [7]:
geo_encoded_df=pd.DataFrame(geo_encoder.toarray(),columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [8]:
## combine one hot encoded columns with original data

data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [9]:
## Save the encoder and scaler

with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)
with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)


In [10]:
# Divide the dataset into independent aand dependent features

# exited is my dependent feature, rem aall are independent features

X=data.drop('Exited',axis=1)
y=data['Exited']

# Split the data in training and testing sets

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

#Scale these features


scaler = StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.fit_transform(X_test)

In [11]:
X_train

array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
         1.72572313, -0.57638802]])

In [12]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)


ANN Implementation

In [13]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

What do 64 and 32 mean?
Dense(64, ...) → 64 neurons in the 1st hidden layer
Dense(32, ...) → 32 neurons in the 2nd hidden layer
🧠 What is a neuron here?

Each neuron:

Takes inputs from previous layer
Applies weights + bias
Passes through activation function (relu here)
🔄 Flow of your model
Input layer (features from X_train)
        ↓
Hidden Layer 1 → 64 neurons (ReLU)
        ↓
Hidden Layer 2 → 32 neurons (ReLU)
        ↓
Output Layer → 1 neuron (Sigmoid)
🎯 Why these numbers?

There’s no fixed rule — it's mostly:

Experimentation
Experience
Problem complexity
General intuition:
More neurons → more learning capacity (but risk of overfitting)
Fewer neurons → simpler model (but may underfit)

In [14]:
## Build our ANN Model
# 1st is 64 neurons then 32
model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)), ### 1st Hideen Layer Connected with input layer
    Dense(32,activation='relu'), ## HL2
    Dense(1,activation='sigmoid') ## output layer
])

What model.summary() is showing

It gives you a layer-by-layer breakdown of:

Output shapes
Number of parameters (weights + biases)
Total trainable parameters
🧱 Let’s decode your output
✅ 1. First Layer
dense (Dense) → Output Shape: (None, 64) → Param #: 832
👉 Meaning:
(None, 64):
None → batch size (can vary)
64 → number of neurons in this layer
🧮 How 832 params are calculated:

Formula:

(input_features × neurons) + biases

So:

= (input_features × 64) + 64

👉 From your result:

832 = (input_features × 64) + 64

Solve:

input_features = 12

✔️ So your dataset has 12 input features

✅ 2. Second Layer
dense_1 (Dense) → Output Shape: (None, 32) → Param #: 2080
🧮 Calculation:
= (64 × 32) + 32
= 2048 + 32
= 2080 ✔️

👉 Because previous layer has 64 neurons

✅ 3. Output Layer
dense_2 (Dense) → Output Shape: (None, 1) → Param #: 33
🧮 Calculation:
= (32 × 1) + 1
= 32 + 1
= 33 ✔️
🔢 Total Parameters
Total params: 2945

👉 Sum of all:

832 + 2080 + 33 = 2945 ✔️
🧠 Trainable vs Non-trainable
Trainable params: 2945
Non-trainable params: 0
Trainable → weights & biases updated during training
Non-trainable → frozen (not used here)
🔥 Key Takeaways (Interview Ready)

You can say:

Output shape (None, X) → batch size + number of neurons
Parameters = (inputs × neurons) + biases
Each neuron has:
One weight per input
One bias
Total params represent model complexity
⚡ Quick Visualization
Input (12 features)
        ↓
Dense (64 neurons) → 832 params
        ↓
Dense (32 neurons) → 2080 params
        ↓
Dense (1 neuron) → 33 params

If you want, I can also explain:
👉 Why too many parameters can cause overfitting and how to control it (very useful for interviews)

In [15]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [16]:
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)

In [17]:
## Compile the model, with multiclass we need to use sparsecrossentropy in loss func, adam has a fixed learning rate, to give it our own learning rate use the code block above


model.compile(optimizer=opt,loss="binary_crossentropy",metrics=['accuracy'])


In [18]:
## Set Up Tensorboard
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard

log_dir="logs/fit" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)


In [19]:
## Set up Early Stopping, we can train a model for different epochs for forward and backward epochs, if we see at 20 epochs if my model has trained its best
## if the variance of the decreaing loss value isn't much then 100 epochs is not required, 20 epochs is only required so we use early stopping

early_stopping_callback=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)



In [20]:
## Train The Model

history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,callbacks=[
        tensorflow_callback, early_stopping_callback
    ]
)

Epoch 1/100


250/250 [==============================] - 6s 9ms/step - loss: 0.3946 - accuracy: 0.8372 - val_loss: 0.3497 - val_accuracy: 0.8535
Epoch 2/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3535 - accuracy: 0.8550 - val_loss: 0.3510 - val_accuracy: 0.8520
Epoch 3/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3484 - accuracy: 0.8605 - val_loss: 0.3460 - val_accuracy: 0.8615
Epoch 4/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3435 - accuracy: 0.8585 - val_loss: 0.3371 - val_accuracy: 0.8650
Epoch 5/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3415 - accuracy: 0.8596 - val_loss: 0.3539 - val_accuracy: 0.8550
Epoch 6/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3376 - accuracy: 0.8615 - val_loss: 0.3479 - val_accuracy: 0.8605
Epoch 7/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3351 - accuracy: 0.8631 - val_loss: 0.3500 - val_accuracy: 0.86

In [21]:
model.save('model.h5')

d:\Gen AI Projects\ANN Project\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [22]:
## Load Tensorboard Extension

%load_ext tensorboard

In [23]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 24432), started 3:43:38 ago. (Use '!kill 24432' to kill it.)

In [25]:
from tensorboard import notebook
notebook.list()

Known TensorBoard instances:
  - port 6008: logdir logs/fit20260328-121835/train (started 3:08:36 ago; pid 10864)
  - port 6006: logdir logs/fit (started 3:44:53 ago; pid 24432)
  - port 6007: logdir logs/fit (started 3:44:36 ago; pid 25340)


In [ ]:
## Load the pickle file

